# Expected Value and Standard Error: Python Functions
**ENGR 1451/2451 · Exploratory Data Science · w07a**

Practice a few NumPy, pandas, Matplotlib, and SciPy functions alongside the slides.
Run the cells in order; keep the `data` folder beside this notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

pd.set_option("display.precision", 4)

## 1. Calculate a mean and SD: `mean()` and `std()`

The six die faces are equally likely. Here, `std(ddof=0)` uses all six population values and divides by 6.

In [ ]:
die_sides = 6
faces = np.arange(1, die_sides + 1)
mu = faces.mean()
sigma = faces.std(ddof=0)

pd.Series({"Population mean": mu, "Population SD": sigma})

## 2. Generate samples: `integers()` and `mean(axis=1)`

`integers()` excludes its upper bound. Each row contains 36 rolls; `mean(axis=1)` averages across that row.
The seed makes the simulation reproducible.

In [ ]:
n = 36
number_of_samples = 10_000
seed = 17
rng = np.random.default_rng(seed)

rolls = rng.integers(1, die_sides + 1, size=(number_of_samples, n))
means = rolls.mean(axis=1)
die_se = sigma / np.sqrt(n)

pd.Series({
    "Mean of simulated means": means.mean(),
    "SD of simulated means": means.std(ddof=0),
    "Known SE of a sample mean": die_se,
})

## 3. Plot the means: `plt.hist()`

Pass the values and bin edges to `hist()`. These bins contain three possible sample means each.

In [ ]:
bin_width = 3 / n
bin_edges = np.arange(means.min() - 0.5 / n,
                      means.max() + bin_width, bin_width)

plt.figure(figsize=(6.5, 3.5))
plt.hist(means, bins=bin_edges, edgecolor="white")
plt.axvline(mu, color="black", linestyle="--")
plt.xlabel(f"Mean of {n} rolls")
plt.ylabel("Number of samples")
plt.show()

## 4. Calculate an interval probability: `stats.norm.cdf()`

Use `loc=mu` and `scale=die_se` for the normal approximation to the sample mean.
Subtract the CDF at 3 from the CDF at 4.

In [ ]:
lower_mean, upper_mean = 3, 4
normal_probability = (
    stats.norm.cdf(upper_mean, loc=mu, scale=die_se)
    - stats.norm.cdf(lower_mean, loc=mu, scale=die_se)
)
print(f"Normal approximation: {normal_probability:.1%}")

## 5. Read real data and estimate SE: `read_csv()` and `stats.sem()`

These 101 aluminum fatigue lives are in **thousand cycles**, measured at 21,000 psi. The population parameters are unknown; treat the measurements as independent observations.

`std(ddof=1)` uses denominator $n-1$. `stats.sem(..., ddof=1)` computes the estimated SE, $s/\sqrt{n}$.

Source: [NIST case study](https://www.itl.nist.gov/div898/handbook/eda/section4/eda4291.htm) · [original data](https://www.itl.nist.gov/div898/handbook/datasets/BIRNSAUN.DAT).

In [ ]:
fatigue = pd.read_csv("data/nist_aluminum_fatigue_21000psi.csv")
lifetime = fatigue["fatigue_life_thousand_cycles"]

pd.Series({
    "Number of specimens": lifetime.size,
    "Mean (thousand cycles)": lifetime.mean(),
    "Sample SD (thousand cycles)": lifetime.std(ddof=1),
    "Estimated SE (thousand cycles)": stats.sem(lifetime, ddof=1),
})

## 6. Short practice
## Die Example ##
1. Set `new_n = 144`. Generate 10,000 new samples of `new_n` die rolls and report the mean of the sample means, their SD, and the theoretical SE $\sigma/\sqrt{n}$.


## Fatigue Example ##
2. Verify with Python that `stats.sem(lifetime, ddof=1)` equals `lifetime.std(ddof=1) / np.sqrt(lifetime.size)`.


In [ ]:
# Add your calculations here.

### General Python function reference

| Purpose | General syntax | Notes |
|---|---|---|
| Create evenly spaced integers | `np.arange(start, stop, step)` | `stop` is excluded. |
| Calculate a mean | `array.mean(axis=None)` | Use `axis=1` for row means in a 2D array. |
| Calculate a standard deviation | `array.std(ddof=0)` | `ddof=0` gives population SD; `ddof=1` is commonly used for sample SD. |
| Square root | `np.sqrt(x)` | Works with scalars and arrays. |
| Create a random-number generator | `rng = np.random.default_rng(seed)` | A fixed seed makes results reproducible. |
| Generate random integers | `rng.integers(low, high, size=...)` | `high` is excluded. |
| Draw a histogram | `plt.hist(values, bins=...)` | `bins` can be a number or an array of bin edges. |
| Add a vertical reference line | `plt.axvline(x, linestyle="--")` | Useful for showing a mean or other reference value. |
| Normal cumulative probability | `stats.norm.cdf(x, loc=mean, scale=sd)` | Returns $P(X \le x)$ for a normal distribution. |
| Number of values in a Series | `series.size` | Counts all elements. |
| Standard error of the mean | `stats.sem(values, ddof=1)` | By default, SciPy uses `ddof=1`. |

Use `help(function_name)` or `function_name?` in Jupyter when you want to inspect a function's arguments and documentation.
